# Introducción a PyTorch

+ PyTorch es una librería de Python de código abierto para desarrollo de productos de ciencia de datos avanzados
  + Deep Learning, Reinforcement Learning, Natural Language Processing (NLP), visión por computadora

+ Fue desarrollada principalmente por el Facebook's AI Research Lab (FAIR)

+ Hoy día la mantienen la Linux Foundation y la comunidad global en general.

In [ ]:
# Instalación
#pip install torch

In [ ]:
import torch

+ Tiene varias (muchas) ventajas en el desarrollo de modelos de DS/IA

+ Por ahora pensemos sólo en dos que son muy importantes para el módulo: trabaja con tensores (estructuras algebraicas con esteroides) y hace derivadas de manera muy eficiente.

+ Empecemos por la parte de derivadas

+ En PyTorch, **Autograd** (Automatic Differentiation) es la parte más importante para esto.

+ Hablando vagamente, cuando se está entrena un modelo de ML, el objetivo es minimizar el error (la función de pérdida o loss)... Luego entonces hacer derivadas entra al juego.

+ Para saber en qué dirección y cuánto ajustar las variables que rigen la lógica de un modelo, se necesita calcular la derivada de la función de pérdida con respecto a cada una de estas variables. Por ejemplos las betas en modelos lineales, las betas en SVM y SVR, los pesos en redes neuronales, los parámetros distribucionales en verosimilitudes ... y un largo etcétera.

+ Tons... ¿qué hacemos?

+ **Propuesta 1:** Derivar las ecuaciones en papel y programar la fórmula de la derivada. Si e modelo es simple en términos de sus parámetros todo parece ok, pero si tiene millones o billones de parámetros (como en redes neuronales), derivar "a mano" es imposible de manejar.

+ **Propuesta 2:** Diferenciación numérica a partir de calcular la derivada aproximada usando incrementos muy pequeños
$$f'(x) \approx \frac{f(x + h) - f(x)}{h}$$
Aunque por supuesto, puede ser imprecisa (por errores de redondeo en computadoras) y potencialamente lenta (requiere evaluar derivadas numéricas para cada variable).

+ **Propuesta 3:** Autograd!!! Aplica la regla de la cadena línea por línea a nivel de código mientras el programa corre. Es exacta, rápida y usa derivadas manuales PERO no propoporcionadas por la cientifica de datos.

+ Para entender por qué se dice que "calcula derivadas exactas usando la regla de la cadena" hay que desmitificar que las computadoras sólo pueden aproximar derivadas mediante $$\frac{\Delta y}{\Delta x},$$
con pasos pequeños.

+ Autograd no aproxima, encuentra el valor exacto hasta el último decimal que permita la precisión flotante de la memoria.

+ **OJO!!!!** No estoy diciendo que la computadora conoce un número infinito de derivadas analíticas. Lo que realmente ocurre es que utiliza composiciones de funciones primitivas.

+ No necesita conocer "todas" las derivadas del universo; sólo necesita conocer las derivadas de un grupo muy reducido de operaciones elementales (que son unas pocas docenas) y las combina... Es decir, ya "se sabe" las derivadas que usábamos en nuestros formularios de la prepa.

+ Autograd descompone cualquier función compleja, por kilométrica que sea, en sus operaciones atómicas elementales (sumas, restas, multiplicaciones, senos, exponenciales) de las cuales la computadora ya conoce la derivada analítica exacta.

+ Visto así... la aportación de PyTorch (aunque no la única) es la descomposición de funciones y el guardado de toda la información que se genera en el camino...

+ Por ejemplo, supóngase que para la función
$$y = \sin(x^2 + 1)$$
se quiere saber la derivada respecto a $x$ evaluada en $x = 2$

+ ¿Qué hace Autograd?

**Paso A:** Descomposición forward. Autograd separa la fórmula en un "grafo de operaciones elementales":

  + $a = x^2$ (para $x=2 \implies a = 4$)
  + $b = a + 1$ (para $a=4 \implies b = 5$)
  + $y = \sin(b)$ para $b=5 \implies y = \sin(5) \approx -0.9589$)

**Paso B:** Autograd consulta las derivadas atómicas guardadas, i.e. "conoce" la fórmula exacta de las derivadas de esas operaciones básicas:
  + La derivada de $x^2$ es $2x$.
  + La derivada de $a + 1$ es $1$.
  + La derivada de $\sin(b)$ es $\cos(b)$.

**Paso C:** Aplica la regla de la cadena hacia atrás (backward)

Cuando se ejecuta la instrucción `y.backward()`, Autograd recorre los pasos al revés multiplicando los resultados intermedios exactos, i.e. usa la regla de la cadena
$$\frac{dy}{dx} = \frac{dy}{db} \cdot \frac{db}{da} \cdot \frac{da}{dx},$$
sustituyendo con las fórmulas que ya conoce
$$\frac{dy}{dx} = \cos(b) \cdot 1 \cdot (2x)$$
y evaluando con los números reales almacenados durante el forward ($x=2$, $b=5$):$$\frac{dy}{dx} = \cos(5) \cdot 1 \cdot (2 \cdot 2) = 0.28366 \cdot 1 \cdot 4 = 1.13464$$

+ Por supuesto, esta lógica no es infalible.

+ Por ejemplo, si se usa una función escalonada (o un if de Python que dependa discontinuamente de los datos sin una pendiente utilizable), la derivada no existe o es $0$ en casi todos lados.

+ Por eso en redes neuronales se suele usar funciones suaves como sigmoide, Tanh o GELU en lugar de saltos abruptos.


In [ ]:
# Se definen tensores con requires_grad = True para trackear los cálculo
x = torch.tensor(2.0, requires_grad = True)
y = torch.tensor(3.0, requires_grad = True)
print("Tensor x:", x)
print("Tensor y:", y)

Tensor x: tensor(2., requires_grad=True)
Tensor y: tensor(3., requires_grad=True)


In [ ]:
# Se define la función z = x^2 + y^3
z = x**4 + y**2
print("Tensor output:", z) # 2^4 + 3^2 = 16 + 9 = 25

Tensor output: tensor(25., grad_fn=<AddBackward0>)


In [ ]:
# Se calculan los gradientes
z.backward()
print("Gradiente de x:", x.grad) # 4x^3 en x=2 es 4(8)=32
print("Gradiente de y:", y.grad) # 2y en y=3 es 2(3)=6

Gradiente de x: tensor(32.)
Gradiente de y: tensor(6.)


+ El ejemplo de $y = \sin(x^2 +1)$ que se describió

In [ ]:
# 1. Se define el valor de x y se le pide a PyTorch que trackee sus gradientes
x = torch.tensor(4.0, requires_grad = True)

# 2. Se definimos la función "matemática" (Forward Pass)
# y = sin(x^2 + 1)
y = torch.sin(x**2 + 1) # sen(4^2 + 1) = sen(17 radianes)
print("El valor de y es:", y)

El valor de y es: tensor(-0.9614, grad_fn=<SinBackward0>)


In [ ]:
# 3. Se ejecutamos Autograd para calcular las derivadas (Backward Pass)
y.backward()

# 4. Consultamos el gradiente acumulado en x (dy/dx)
print("El gradiente dy/dx es:", x.grad.item())

El gradiente dy/dx es: -2.2013068199157715


+ PyTorch usa el concepto de grafo computacional.

+ Suena muy fancy, pero es una abstracción que representa un diagrama o estructura de datos formado por:

  + Nodos: Representan variables (datos de entrada, constantes) u operaciones (sumas, multiplicaciones, senos, logaritmos).
  + Aristas: Representan el flujo de información (de dónde viene el valor y hacia qué operación va).

+ Representa a las expresiones matemáticas en grafos computacionales.

+ De hecho, usa una evolución de esta abstracción: el grafo computacional dinámico (a.k.a Define-by-Run).

## Grafo estático (Define-and-Run)

+ Es el enfoque antiguo.

+ Sistemas tradicionales (como TensorFlow viejito (1.x)) lo usaban, obligando a trabajar en dos fases separadas:

  + Fase 1 (Diseño): Se construyes la "estructura matemática" completa en memoria. En esta fase no hay datos reales fluyendo, sólo se define la "tubería".

  + Fase 2 (Ejecución): Se creas una "sesión" y se deja caer los datos por las tuberías que se construyeron.

+ Obviamente, si se quiere cambiar algo a mitad de camino, es extremadamente complejo porque el plano ya está "congelado".

+ Además, si algo falla, los errores eran difíciles de identificar porque ocurren dentro de la "sesión", no en el código de Python.

## Grafo dinámico (Define-by-Run)

+ Es el enfoque que usa PyTorch.

+ Construye el grafo paso a paso a medida que se ejecuta el código línea por línea en Python.

+ Cada vez que se realiza un cálculo con un tensor (una suma, una multiplicación), PyTorch anota en segundo plano esa operación.

+ Al terminar el forward pass, el grafo se ha creado sólo para esa ejecución, se utiliza para calcular las derivadas y luego se destruye para volver a crearse en la siguiente iteración.

+ **OJO:** La estructura y los variables objetivo son lo que perdura y se optimiza). Esto NUNCA se destruye. De hecho, es lo que el optimizador modifica en cada paso para que el modelo sea cada vez más inteligente.

  + El grafo computacional es sólo un historial temporal de operaciones matemáticas que le sirve a Autograd como "mapa de ruta" para saber qué camino tomar al calcular las derivadas.

## Uso en una red neuroanl

+ Ver videito: https://youtu.be/QetoD5LXlEg?si=zPs7ulh5OE0EuKXU